# Mineral-map component-weight optimization

This notebook is a separate, reproducible workflow for finding non-reductive additive component weights. It keeps the scientific distance model and the exploratory force-directed layout distinct: candidate weights are evaluated against normalized component behaviour, graph stability, component diversity, and soft classification continuity before one deterministic candidate is exported.

The source dataset is the existing 6,228-mineral IMA collection. Geographic-map concepts from the generic project outline are adapted here to the mineral-composition graph: coordinates are 3D layout coordinates, regions are compositional communities, and map features are browser node records.


In [1]:
# 1. Set up the new notebook and dependencies
# This cell defines paths, reproducibility settings, component names, the
# chemistry-first prior, and output locations.
import csv
import json
import math
import random
from collections import Counter
from pathlib import Path
from typing import Any

import networkx as nx
import numpy as np

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)

# The notebook may be launched from the workspace root or from another working
# directory. Resolve project files from the notebook's own location where possible.
NOTEBOOK_ROOT = Path.cwd()
ROOT = NOTEBOOK_ROOT
if not (ROOT / "mineral_visualization_edges.csv").is_file() and (ROOT.parent / "mineral_visualization_edges.csv").is_file():
    ROOT = ROOT.parent

COMPONENT_NAMES = (
    "anion_group",
    "cations",
    "extra_anions",
    "hydration",
    "structural_water",
    "structure",
)
BASELINE_WEIGHTS = np.array([0.43, 0.33, 0.12, 0.05, 0.05, 0.02], dtype=np.float64)
assert np.isclose(BASELINE_WEIGHTS.sum(), 1.0)

MINERAL_CSV = ROOT / "IMA_data_with_derived_strunz.csv"
EDGE_CSV = ROOT / "mineral_visualization_edges.csv"
WEB_NODES = ROOT / "web_export" / "mineral-map-nodes.json"
WEB_METADATA = ROOT / "web_export" / "mineral-map-metadata.json"
FRONTEND_NODES = ROOT / "mineral-map" / "public" / "data" / "mineral-map-nodes.json"
FRONTEND_METADATA = ROOT / "mineral-map" / "public" / "data" / "mineral-map-metadata.json"
REPORT_PATH = ROOT / "component_weight_optimization_report.json"

CANDIDATE_COUNT = 512
PAIR_SAMPLE_SIZE = 100_000
STABILITY_REPLICATES = 4
GRAPH_NEIGHBORS = 10
FORCE_ITERATIONS = 200
FORCE_K = 0.12

# 2. Load and validate source data
# Reuse the authoritative component explanations from the exact graph export.
def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Required input does not exist: {path}")

for path in (MINERAL_CSV, EDGE_CSV, WEB_NODES, WEB_METADATA):
    require_file(path)

with MINERAL_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    mineral_rows = list(csv.DictReader(input_file))
with EDGE_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    edge_rows = list(csv.DictReader(input_file))
with WEB_NODES.open(encoding="utf-8") as input_file:
    web_payload = json.load(input_file)
with WEB_METADATA.open(encoding="utf-8") as input_file:
    metadata_payload = json.load(input_file)

required_edge_columns = {"source_index", "target_index", "distance", *[f"raw_{name}" for name in COMPONENT_NAMES]}
missing_edge_columns = required_edge_columns - set(edge_rows[0]) if edge_rows else required_edge_columns
if missing_edge_columns:
    raise ValueError(f"Edge export is missing required columns: {sorted(missing_edge_columns)}")
if len(mineral_rows) != len(web_payload.get("nodes", [])):
    raise ValueError("Mineral CSV and web-node export have different row counts.")

node_count = len(mineral_rows)
print(f"Loaded {node_count:,} minerals and {len(edge_rows):,} directed k-NN edges.")
print(f"Input metadata coordinate system: {metadata_payload.get('coordinateSystem', 'unknown')}")

# 3. Implement data cleaning and normalization
edge_array = np.asarray(
    [[float(row["source_index"]), float(row["target_index"]), *[float(row[f"raw_{name}"]) for name in COMPONENT_NAMES]] for row in edge_rows],
    dtype=np.float64,
)
edge_sources = edge_array[:, 0].astype(np.int32)
edge_targets = edge_array[:, 1].astype(np.int32)
raw_components = edge_array[:, 2:]
if not np.isfinite(raw_components).all() or (raw_components < 0).any():
    raise ValueError("Raw edge component values must be finite and non-negative.")
component_low = np.percentile(raw_components, 5, axis=0)
component_high = np.percentile(raw_components, 95, axis=0)
component_scale = np.maximum(component_high - component_low, 1e-9)
normalized_components = np.clip((raw_components - component_low) / component_scale, 0.0, 1.0)
normalized_components /= np.maximum(np.median(normalized_components, axis=0), 1e-9)
normalized_components = np.clip(normalized_components, 0.0, 4.0)
print("Robust component normalization completed.")

# 4. Build the mineral-map data model
def weighted_edge_distances(weights: np.ndarray, values: np.ndarray = normalized_components) -> np.ndarray:
    weights = np.asarray(weights, dtype=np.float64)
    if weights.shape != (len(COMPONENT_NAMES),) or np.any(weights < 0) or not np.isclose(weights.sum(), 1.0):
        raise ValueError("Weights must be a non-negative six-component vector summing to one.")
    return values @ weights


def graph_from_distances(distances: np.ndarray, sources: np.ndarray = edge_sources, targets: np.ndarray = edge_targets) -> list[set[int]]:
    neighbours = [set() for _ in range(node_count)]
    order = np.lexsort((targets, distances, sources))
    counts = np.zeros(node_count, dtype=np.int8)
    for index in order:
        source = int(sources[index])
        if counts[source] >= GRAPH_NEIGHBORS:
            continue
        neighbours[source].add(int(targets[index]))
        counts[source] += 1
    return neighbours


def graph_jaccard(left: list[set[int]], right: list[set[int]]) -> float:
    scores = []
    for first, second in zip(left, right):
        union = first | second
        scores.append(len(first & second) / len(union) if union else 1.0)
    return float(np.mean(scores))


def effective_contributions(weights: np.ndarray, values: np.ndarray = normalized_components) -> np.ndarray:
    contributions = values * weights[None, :]
    totals = contributions.sum(axis=1)
    return contributions.sum(axis=0) / max(float(totals.sum()), 1e-12)


def entropy(weights: np.ndarray) -> float:
    return float(-np.sum(np.where(weights > 0, weights * np.log(weights), 0.0)) / np.log(len(weights)))


def component_redundancy(weights: np.ndarray) -> float:
    correlations = []
    for left in range(len(COMPONENT_NAMES)):
        for right in range(left + 1, len(COMPONENT_NAMES)):
            correlation = np.corrcoef(normalized_components[:, left], normalized_components[:, right])[0, 1]
            if np.isfinite(correlation):
                correlations.append(abs(float(correlation)))
    return float(np.mean(correlations))

# 5. Implement geographic filtering and aggregation
# In the mineral graph, compositional regions replace geographic regions.
def filter_mineral_indices(query: str = "", strunz_code: str | None = None, bounds: tuple[np.ndarray, np.ndarray] | None = None) -> np.ndarray:
    query = query.casefold().strip()
    coordinates = np.asarray([node["coordinates"] for node in web_payload["nodes"]], dtype=np.float64)
    selected = []
    for index, row in enumerate(mineral_rows):
        if query and query not in f"{row.get('Mineral Name', '')} {row.get('Valence Chemistry (concise)', '')}".casefold():
            continue
        if strunz_code and row.get("Derived Strunz Top-Level") != strunz_code:
            continue
        if bounds is not None and not (np.all(coordinates[index] >= bounds[0]) and np.all(coordinates[index] <= bounds[1])):
            continue
        selected.append(index)
    return np.asarray(selected, dtype=np.int32)


def aggregate_indices(indices: np.ndarray) -> dict[str, Any]:
    if len(indices) == 0:
        return {"count": 0, "centroid": None, "strunz_counts": {}}
    coordinates = np.asarray([web_payload["nodes"][int(index)]["coordinates"] for index in indices])
    return {"count": int(len(indices)), "centroid": coordinates.mean(axis=0).tolist(), "strunz_counts": dict(Counter(mineral_rows[int(index)].get("Derived Strunz Top-Level", "") for index in indices))}

# 6. Create the interactive-map export model
def normalized_force_layout(weights: np.ndarray) -> tuple[np.ndarray, nx.Graph]:
    distances = weighted_edge_distances(weights)
    graph = nx.Graph()
    graph.add_nodes_from(range(node_count))
    pairs: dict[tuple[int, int], float] = {}
    for distance, source, target in zip(distances, edge_sources, edge_targets):
        pair = tuple(sorted((int(source), int(target))))
        pairs[pair] = min(float(distance), pairs.get(pair, math.inf))
    values = np.asarray(list(pairs.values()))
    low, high = np.percentile(values, [5, 95])
    for (source, target), distance in pairs.items():
        similarity = 1.0 - np.clip((distance - low) / max(high - low, 1e-9), 0.0, 1.0)
        graph.add_edge(source, target, weight=float(0.2 + 0.8 * similarity))
    initial = np.asarray([node["coordinates"] for node in web_payload["nodes"]], dtype=np.float64)
    initial -= initial.mean(axis=0, keepdims=True)
    initial /= max(np.max(np.linalg.norm(initial, axis=1)), 1e-9)
    positions = nx.spring_layout(graph, dim=3, seed=SEED, iterations=FORCE_ITERATIONS, k=FORCE_K, scale=1.0, weight="weight", pos={i: initial[i] for i in range(node_count)})
    coordinates = np.asarray([positions[i] for i in range(node_count)], dtype=np.float64)
    coordinates -= coordinates.mean(axis=0, keepdims=True)
    coordinates /= max(np.max(np.abs(coordinates)), 1e-9)
    return coordinates * 0.95, graph

# 7. Add user controls and event handling
DIRICHLET_CONCENTRATION = 80.0
MIN_EFFECTIVE_CONTRIBUTION = 0.025
MAX_DOMINANT_CONTRIBUTION = 0.62
MIN_ACTIVE_COMPONENTS = 3
MIN_STABILITY = 0.35
candidate_weights = np.vstack([BASELINE_WEIGHTS, rng.dirichlet(BASELINE_WEIGHTS * DIRICHLET_CONCENTRATION, size=CANDIDATE_COUNT)])

# 8. Connect the map to the application data
# Evaluate graph stability, effective contributions, redundancy, and soft Strunz continuity.
def evaluate_candidate(weights: np.ndarray, candidate_index: int) -> dict[str, Any]:
    distances = weighted_edge_distances(weights)
    graph = graph_from_distances(distances)
    contribution = effective_contributions(weights)
    active_components = int(np.sum(contribution >= MIN_EFFECTIVE_CONTRIBUTION))
    perturbation_scores = []
    for _ in range(STABILITY_REPLICATES):
        perturbation = rng.normal(0.0, 0.04, size=len(weights))
        perturbed = np.clip(weights * np.exp(perturbation), 1e-6, None)
        perturbed /= perturbed.sum()
        perturbation_scores.append(graph_jaccard(graph, graph_from_distances(weighted_edge_distances(perturbed))))
    sampled_indices = rng.choice(len(edge_rows), size=min(PAIR_SAMPLE_SIZE, len(edge_rows)), replace=False)
    subsample_stability = graph_jaccard(graph, graph_from_distances(distances[sampled_indices], edge_sources[sampled_indices], edge_targets[sampled_indices]))
    source_sample = rng.choice(node_count, size=min(256, node_count), replace=False)
    class_scores = []
    for source in source_sample:
        label = mineral_rows[int(source)].get("Derived Strunz Top-Level", "")
        class_scores.append(np.mean([mineral_rows[target].get("Derived Strunz Top-Level", "") == label for target in graph[int(source)]]) if graph[int(source)] else 0.0)
    classification_agreement = float(np.mean(class_scores))
    dominant = int(np.argmax(contribution))
    reasons = []
    if active_components < MIN_ACTIVE_COMPONENTS:
        reasons.append("too_few_active_components")
    if contribution[dominant] > MAX_DOMINANT_CONTRIBUTION:
        reasons.append("dominant_component")
    if float(np.mean(perturbation_scores)) < MIN_STABILITY:
        reasons.append("low_weight_stability")
    score = 0.35 * float(np.mean(perturbation_scores)) + 0.20 * subsample_stability + 0.15 * classification_agreement + 0.20 * entropy(weights) + 0.10 * (1.0 - component_redundancy(weights))
    return {"candidate_index": candidate_index, "weights": dict(zip(COMPONENT_NAMES, map(float, weights))), "effective_contributions": dict(zip(COMPONENT_NAMES, map(float, contribution))), "active_components": active_components, "dominant_component": COMPONENT_NAMES[dominant], "entropy": entropy(weights), "weight_stability": float(np.mean(perturbation_scores)), "subsample_stability": float(subsample_stability), "strunz_local_agreement": classification_agreement, "component_redundancy": component_redundancy(weights), "score": float(score), "accepted": not reasons, "rejection_reasons": reasons}

# 9. Add error handling and input validation
candidate_reports = [evaluate_candidate(weights, index) for index, weights in enumerate(candidate_weights)]
accepted_reports = sorted((report for report in candidate_reports if report["accepted"]), key=lambda report: (-report["score"], report["candidate_index"]))
if not accepted_reports:
    raise RuntimeError("No non-reductive candidate weights survived. Relax thresholds or broaden the prior.")
selected_report = accepted_reports[0]
selected_weights = np.asarray([selected_report["weights"][name] for name in COMPONENT_NAMES], dtype=np.float64)
print(f"Accepted {len(accepted_reports):,} of {len(candidate_reports):,} candidate weight sets.")
print("Selected weights:", selected_report["weights"])
print("Effective contributions:", selected_report["effective_contributions"])

# 10. Test core functions
assert node_count == 6228
assert np.isclose(selected_weights.sum(), 1.0)
assert np.all(selected_weights >= 0)
assert np.isfinite(normalized_components).all()
assert len(graph_from_distances(weighted_edge_distances(selected_weights))) == node_count
assert len(filter_mineral_indices()) == node_count
assert aggregate_indices(np.array([], dtype=np.int32))["count"] == 0
print("Core tests passed.")

# 11. Run an end-to-end implementation check
selected_coordinates, selected_graph = normalized_force_layout(selected_weights)
assert selected_coordinates.shape == (node_count, 3)
assert np.isfinite(selected_coordinates).all()
assert np.max(np.abs(selected_coordinates)) <= 0.950001
representative = filter_mineral_indices(query="quartz")
assert len(representative) > 0
print(f"End-to-end check passed: quartz filter returned {len(representative)} minerals.")
print(f"Selected graph: {selected_graph.number_of_nodes():,} nodes, {selected_graph.number_of_edges():,} undirected edges.")

# 12. Export processed data and configuration
optimization_report = {"schemaVersion": "component-weight-optimization-v1", "selectionPolicy": "highest composite score among candidates passing non-reductive constraints; deterministic tie-break by candidate index", "seed": SEED, "componentNames": list(COMPONENT_NAMES), "baselineWeights": dict(zip(COMPONENT_NAMES, map(float, BASELINE_WEIGHTS))), "selectedCandidate": selected_report, "thresholds": {"minimumEffectiveContribution": MIN_EFFECTIVE_CONTRIBUTION, "maximumDominantContribution": MAX_DOMINANT_CONTRIBUTION, "minimumActiveComponents": MIN_ACTIVE_COMPONENTS, "minimumStability": MIN_STABILITY}, "candidateCount": len(candidate_reports), "acceptedCount": len(accepted_reports), "topCandidates": accepted_reports[:20]}
REPORT_PATH.write_text(json.dumps(optimization_report, indent=2) + "\n", encoding="utf-8")
with WEB_NODES.open(encoding="utf-8") as input_file:
    export_nodes = json.load(input_file)
with WEB_METADATA.open(encoding="utf-8") as input_file:
    export_metadata = json.load(input_file)
for node, coordinates in zip(export_nodes["nodes"], selected_coordinates):
    node["coordinates"] = [float(value) for value in coordinates]
export_nodes["coordinateSystem"] = "normalized 3D force-directed layout using automatically selected additive component weights"
export_metadata["componentWeights"] = selected_report["weights"]
export_metadata["distanceModel"]["mapAlgorithm"] = "weighted 3D force-directed k-NN graph"
export_metadata["distanceModel"]["mapParameters"] = {"dimensions": 3, "iterations": FORCE_ITERATIONS, "k": FORCE_K, "randomState": SEED, "undirectedEdges": selected_graph.number_of_edges(), "weightSelectionReport": REPORT_PATH.name}
export_metadata["distanceModel"]["coordinateCaveat"] = "Normalized force-directed coordinates are exploratory; exact edge distances and component values remain authoritative."
for path, payload in ((WEB_NODES, export_nodes), (WEB_METADATA, export_metadata), (FRONTEND_NODES, export_nodes), (FRONTEND_METADATA, export_metadata)):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=True, separators=(",", ":")) + "\n", encoding="utf-8")
print(f"Wrote {REPORT_PATH} and exported {node_count:,} selected coordinates.")


Loaded 6,228 minerals and 62,280 directed k-NN edges.
Input metadata coordinate system: normalized 3D force-directed layout of the exact additive k-nearest-neighbour graph; edge distances remain authoritative
Robust component normalization completed.
Accepted 512 of 513 candidate weight sets.
Selected weights: {'anion_group': 0.3102713920901254, 'cations': 0.3223567163838267, 'extra_anions': 0.15809244174817924, 'hydration': 0.0906987675275306, 'structural_water': 0.07976691488131664, 'structure': 0.038813767369021236}
Effective contributions: {'anion_group': 0.1330986148101503, 'cations': 0.3940481865907539, 'extra_anions': 0.3022050863724447, 'hydration': 0.1090715470789679, 'structural_water': 0.028825596496422207, 'structure': 0.032750968651156405}
Core tests passed.
End-to-end check passed: quartz filter returned 1 minerals.
Selected graph: 6,228 nodes, 43,198 undirected edges.
Wrote c:\My Things\HTML\Demonin's Item Shop\other\minerals\component_weight_optimization_report.json and

In [5]:
nearest_neighbors = NearestNeighbors(
    n_neighbors=min(node_count, GRAPH_NEIGHBORS * 8 + 1),
    metric="euclidean",
    algorithm="auto",
    n_jobs=-1,
)
nearest_neighbors.fit(weighted_landmark_signatures)
approximate_distances, approximate_indices = nearest_neighbors.kneighbors(weighted_landmark_signatures)

# Build a fresh directed graph from the new full 6,228-mineral search. Query
# extra candidates because duplicate landmark signatures can occupy slots.
approximate_edges = []
for source in range(node_count):
    rank = 0
    seen_targets = {source}
    for column in range(1, approximate_indices.shape[1]):
        target = int(approximate_indices[source, column])
        if target in seen_targets:
            continue
        seen_targets.add(target)
        rank += 1
        approximate_edges.append({
            "source_index": source,
            "target_index": target,
            "neighbor_rank": rank,
            "distance": float(approximate_distances[source, column]),
        })
        if rank == GRAPH_NEIGHBORS:
            break
    if rank != GRAPH_NEIGHBORS:
        raise RuntimeError(f"Could not find {GRAPH_NEIGHBORS} unique neighbours for source {source}; found {rank}.")

if len(approximate_edges) != node_count * GRAPH_NEIGHBORS:
    raise RuntimeError(f"Expected {node_count * GRAPH_NEIGHBORS:,} approximate edges, got {len(approximate_edges):,}.")


In [3]:
with np.load(ROOT / "additive_component_distances_256.npz", allow_pickle=False) as cache:
    print("256-landmark cache keys:", cache.files)
    print("256-landmark cache shapes:", {key: cache[key].shape for key in cache.files})
print("component names:", COMPONENT_NAMES)


256-landmark cache keys: ['anion_group', 'cations', 'extra_anions', 'hydration', 'structure']
256-landmark cache shapes: {'anion_group': (6228, 256), 'cations': (6228, 256), 'extra_anions': (6228, 256), 'hydration': (6228, 256), 'structure': (6228, 256)}
component names: ('anion_group', 'cations', 'extra_anions', 'hydration', 'structural_water', 'structure')


In [7]:
# Continue the 256-landmark regeneration: layout, export, and diagnostics.
force_graph = nx.Graph()
force_graph.add_nodes_from(range(node_count))
edge_distance_values = np.asarray([edge["distance"] for edge in approximate_edges], dtype=np.float64)
edge_low, edge_high = np.percentile(edge_distance_values, [5, 95])
edge_lookup = {}
for edge in approximate_edges:
    pair = tuple(sorted((int(edge["source_index"]), int(edge["target_index"]))))
    similarity = 1.0 - np.clip((edge["distance"] - edge_low) / max(edge_high - edge_low, 1e-9), 0.0, 1.0)
    edge_lookup[pair] = max(float(0.2 + 0.8 * similarity), edge_lookup.get(pair, 0.0))
for (source, target), weight in edge_lookup.items():
    force_graph.add_edge(source, target, weight=weight)

fresh_initial = rng.normal(0.0, 0.35, size=(node_count, 3))
fresh_initial -= fresh_initial.mean(axis=0, keepdims=True)
fresh_initial /= max(np.max(np.linalg.norm(fresh_initial, axis=1)), 1e-9)
force_positions = nx.spring_layout(
    force_graph, dim=3, seed=SEED, iterations=FORCE_ITERATIONS, k=FORCE_K,
    scale=1.0, weight="weight", pos={index: fresh_initial[index] for index in range(node_count)},
)
regenerated_coordinates = np.asarray([force_positions[index] for index in range(node_count)], dtype=np.float64)
regenerated_coordinates -= regenerated_coordinates.mean(axis=0, keepdims=True)
regenerated_coordinates /= max(np.max(np.abs(regenerated_coordinates)), 1e-9)
regenerated_coordinates *= 0.95

with WEB_NODES.open(encoding="utf-8") as input_file:
    regenerated_nodes_payload = json.load(input_file)
with WEB_METADATA.open(encoding="utf-8") as input_file:
    regenerated_metadata = json.load(input_file)
for node, coordinates in zip(regenerated_nodes_payload["nodes"], regenerated_coordinates):
    node["coordinates"] = [float(value) for value in coordinates]
regenerated_nodes_payload["coordinateSystem"] = "normalized 3D force-directed layout from regenerated 256-landmark weighted k-NN signatures"
regenerated_metadata["componentWeights"] = dict(zip(COMPONENT_NAMES, map(float, selected_weights)))
regenerated_metadata["distanceModel"]["mapAlgorithm"] = "weighted 3D force-directed k-NN graph with 256-landmark signatures"
regenerated_metadata["distanceModel"]["mapParameters"] = {
    "dimensions": 3, "landmarkCount": LANDMARK_COUNT, "neighborsPerMineral": GRAPH_NEIGHBORS,
    "iterations": FORCE_ITERATIONS, "k": FORCE_K, "randomState": SEED,
    "undirectedEdges": force_graph.number_of_edges(), "weightSelectionReport": REPORT_PATH.name,
}
regenerated_metadata["distanceModel"]["approximation"] = {
    "method": "Euclidean nearest-neighbour search on weighted component-to-landmark signatures",
    "missingCachedComponents": sorted(missing_components),
    "exactPairwiseDistancesComputed": False,
    "warning": "Exploratory 256-landmark topology; structural_water contributes zero because it is absent from the legacy cache.",
}

approximate_neighbors_by_source = [[] for _ in range(node_count)]
for edge in approximate_edges:
    approximate_neighbors_by_source[int(edge["source_index"])].append({
        "targetId": int(edge["target_index"]), "rank": int(edge["neighbor_rank"]),
        "distance": float(edge["distance"]), "category": "approximate_landmark_neighbour",
        "components": {component: {"raw": 0.0, "weighted": 0.0} for component in COMPONENT_NAMES},
    })
regenerated_neighbor_payload = {
    "schemaVersion": "mineral-map-static-v1", "neighborCount": GRAPH_NEIGHBORS,
    "approximate": True, "neighborsBySourceId": approximate_neighbors_by_source,
}
with APPROX_NEIGHBOR_PATH.open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=["source_index", "target_index", "neighbor_rank", "distance"])
    writer.writeheader(); writer.writerows(approximate_edges)
for path, payload in (
    (WEB_NODES, regenerated_nodes_payload), (WEB_METADATA, regenerated_metadata),
    (FRONTEND_NODES, regenerated_nodes_payload), (FRONTEND_METADATA, regenerated_metadata),
    (WEB_NEIGHBORS, regenerated_neighbor_payload), (FRONTEND_NEIGHBORS, regenerated_neighbor_payload),
):
    path.write_text(json.dumps(payload, ensure_ascii=True, separators=(",", ":")) + "\n", encoding="utf-8")
print(f"Regenerated {len(approximate_edges):,} directed edges; {force_graph.number_of_edges():,} undirected edges.")


Regenerated 62,280 directed edges; 42,366 undirected edges.


In [ ]:
# 15. Reapply the original relationship taxonomy to the exported 10-NN neighbours
# Self-contained: it needs only the two exported neighbour payloads and the
# authoritative edge CSV, so it runs even after a kernel restart and touches
# ONLY the 62,280 k-NN edges the frontend displays -- never all-pairs distances.
import csv
import json
from collections import Counter

APPROXIMATE_CATEGORY = "approximate_landmark_neighbour"
APPROXIMATE_DESCRIPTION = (
    "Nearest neighbour found by the exploratory 256-landmark signature search; "
    "relationship class is pending exact pairwise recomputation."
)

with WEB_METADATA.open(encoding="utf-8") as input_file:
    metadata_payload = json.load(input_file)
taxonomy_descriptions = dict(metadata_payload.get("relationshipCategories", {}))
taxonomy_descriptions.setdefault(APPROXIMATE_CATEGORY, APPROXIMATE_DESCRIPTION)

# One linear scan of the 13 MB edge CSV (about a second) to recover the taxonomy
# label that was stored with every original directed edge.
old_category_by_pair = {}
with EDGE_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        category = row.get("relationship_category", "")
        if category:
            old_category_by_pair[(int(row["source_index"]), int(row["target_index"]))] = category

with WEB_NEIGHBORS.open(encoding="utf-8") as input_file:
    neighbor_payload = json.load(input_file)

category_counts = Counter()
for source_id, source_neighbors in enumerate(neighbor_payload["neighborsBySourceId"]):
    for entry in source_neighbors:
        old_category = old_category_by_pair.get((source_id, int(entry["targetId"])))
        entry["category"] = old_category or APPROXIMATE_CATEGORY
        category_counts[entry["category"]] += 1

regenerated_metadata["relationshipCategories"] = taxonomy_descriptions
for path, payload in (
    (WEB_METADATA, regenerated_metadata),
    (FRONTEND_METADATA, regenerated_metadata),
    (WEB_NEIGHBORS, neighbor_payload),
    (FRONTEND_NEIGHBORS, neighbor_payload),
):
    path.write_text(json.dumps(payload, ensure_ascii=True, separators=(",", ":")) + "\n", encoding="utf-8")

print(f"Relabelled {sum(category_counts.values()):,} k-NN edges:")
for category, count in category_counts.most_common():
    print(f"  {category}: {count:,}")


In [8]:
# 14. Quantify whether the regenerated topology actually changed
old_by_source = [[] for _ in range(node_count)]
with EDGE_CSV.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        old_by_source[int(row["source_index"])].append(int(row["target_index"]))
new_by_source = [[] for _ in range(node_count)]
with APPROX_NEIGHBOR_PATH.open(newline="", encoding="utf-8-sig") as input_file:
    for row in csv.DictReader(input_file):
        new_by_source[int(row["source_index"])].append(int(row["target_index"]))

per_source_overlap = np.asarray([
    len(set(old_targets) & set(new_targets)) / max(len(set(old_targets) | set(new_targets)), 1)
    for old_targets, new_targets in zip(old_by_source, new_by_source)
])
per_source_recall = np.asarray([
    len(set(old_targets) & set(new_targets)) / max(len(set(old_targets)), 1)
    for old_targets, new_targets in zip(old_by_source, new_by_source)
])
old_pairs = {tuple(sorted((source, target))) for source, targets in enumerate(old_by_source) for target in targets}
new_pairs = {tuple(sorted((source, target))) for source, targets in enumerate(new_by_source) for target in targets}
print(f"Directed neighbour-set Jaccard: {per_source_overlap.mean():.4f}")
print(f"Directed old-neighbour recall: {per_source_recall.mean():.4f}")
print(f"Directed edges retained: {sum(len(set(old) & set(new)) for old, new in zip(old_by_source, new_by_source)):,} / {node_count * GRAPH_NEIGHBORS:,}")
print(f"Undirected edge-set Jaccard: {len(old_pairs & new_pairs) / max(len(old_pairs | new_pairs), 1):.4f}")
print(f"Sources with at least one changed neighbour: {sum(set(old) != set(new) for old, new in zip(old_by_source, new_by_source)):,} / {node_count:,}")


Directed neighbour-set Jaccard: 0.2784
Directed old-neighbour recall: 0.3939
Directed edges retained: 24,535 / 62,280
Undirected edge-set Jaccard: 0.2500
Sources with at least one changed neighbour: 6,205 / 6,228
